In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import pickle

print("The libraries are imported!!!!!!!")

In [ ]:
dataset = pd.read_csv("/kaggle/input/datasets/shivamshinde123/autismprediction/train.csv")
dataset.head()

In [ ]:
dataset.shape

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset.info()

In [ ]:
# convertinf the age float to int
dataset['age'] = dataset['age'].astype(int)
dataset['age'].head()

In [ ]:
for col in dataset.columns:
    numerical_features = ['ID', 'age', 'result']
    if col not in numerical_features:
        print(col, dataset[col].unique())
        print("-"*50)
        

In [ ]:
dataset = dataset.drop(columns=['ID', 'age_desc'])
dataset.shape

In [ ]:
# define the mapping the dictionary for mapping

mapping = {
    "Viet Nam" : "Vietnam",
    "AmericanSamoa" : "United States",
    "Hong Kong" : "China"
}

# replace the values to new values

dataset['contry_of_res '] = dataset['contry_of_res'].replace(mapping)

# find the unique values again

dataset['contry_of_res '].unique()

In [ ]:
# checking for class imbalance

dataset['Class/ASD'].value_counts()

# lot of imbalance

In [ ]:
dataset.describe()

Univariant analysis
    • categorical
    • numerical

In [ ]:
# numerical data {age, result}

sns.set_theme(style='whitegrid')

plt.figure(figsize=(6,6))
sns.histplot(dataset['age'], kde=True)
plt.title("Distribution of the AGE")
plt.xlabel("Age")
plt.ylabel("Number of people")


# calculating the mean and median values for age

mean_age = dataset['age'].mean()
median_age = dataset['age'].median()

print("The average age of the people:", mean_age)
print("The median age of the people:",median_age)

# displaying the mean and median on the plot

plt.axvline(mean_age, linestyle="--", color='red',label='Mean')
plt.axvline(median_age, linestyle="-", color='green',label='Median')
plt.legend()

plt.show()

# As the dataset is right skew mean > median
# if the dataset is left skew mean < median

In [ ]:
# numerical data {age, result}

sns.set_theme(style='whitegrid')

plt.figure(figsize=(6,6))
sns.histplot(dataset['result'], kde=True)
plt.title("Distribution of the RESULT")
plt.xlabel("Result")
plt.ylabel("Number of people")


# calculating the mean and median values for result

mean_result = dataset['result'].mean()
median_result = dataset['result'].median()

print("The average age of the people:", mean_age)
print("The median age of the people:",median_age)

# displaying the mean and median on the plot

plt.axvline(mean_result, linestyle="--", color='red',label='Mean')
plt.axvline(median_result, linestyle="-", color='green',label='Median')
plt.legend()

plt.show()

# As the dataset is right skew mean > median
# if the dataset is left skew mean < median

In [ ]:
# create subplot layout (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot for age
sns.boxplot(x=dataset['age'], ax=axes[0])
axes[0].set_title("Box Plot for Age")
axes[0].set_xlabel("Age")

# Boxplot for result
sns.boxplot(x=dataset['result'], ax=axes[1])
axes[1].set_title("Box Plot for Result")
axes[1].set_xlabel("Result")

# adjust spacing
plt.tight_layout()

# show plots
plt.show()

In [ ]:
q1 = dataset['age'].quantile(0.25)
q3 = dataset['age'].quantile(0.75)
IQR = q3 - q1
lower_bound = q1 - 1.5 * IQR
upper_bound = q3 + 1.5 * IQR
age_outliers = dataset[(dataset['age'] < lower_bound) | (dataset['age'] > upper_bound)]
print("The number of outliers are:", len(age_outliers))

In [ ]:
q1 = dataset['result'].quantile(0.25)
q3 = dataset['result'].quantile(0.75)
IQR = q3 - q1
lower_bound = q1 - 1.5 * IQR
upper_bound = q3 + 1.5 * IQR
age_outliers = dataset[(dataset['result'] < lower_bound) | (dataset['result'] > upper_bound)]
print("The number of outliers are:", len(age_outliers))

In [ ]:
dataset.columns

In [ ]:
categorical_data = {
    'A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score',
       'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 'gender',
       'ethnicity', 'jaundice', 'austim', 'contry_of_res', 'used_app_before',
        'relation', 'Class/ASD',
}

cols = 3
rows = math.ceil(len(categorical_data)/cols)
plt.figure(figsize=(15, 5 * rows))
for i, col in enumerate(categorical_data, 1):
    plt.subplot(rows,cols,i)

    sns.countplot(x=dataset[col])
    plt.title(col)
    plt.xticks(rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
dataset['Class/ASD'].value_counts()

handling missing values in ethnicity and relation columns

In [ ]:
dataset['relation'].unique()

In [ ]:
dataset['ethnicity'].replace({
    "?" : "others",
    "Others" : "others"
}, inplace=True)

dataset['relation'].replace(
    {
        "Relative" : "others",
        "Parent" : "others",
        "?" : "others",
        "Others" : "others",
        "Health care professional" : "others"
    }, inplace=True
)

print(dataset['ethnicity'].unique())
print(dataset['relation'].unique())

In [ ]:
# obeject cloumns

object_columns = dataset.select_dtypes(include=["object"]).columns

# intialize dictionary for storing the encoders

encoded_dict = { }

for col in object_columns:
    encoder = LabelEncoder()
    dataset[col] = encoder.fit_transform(dataset[col])
    encoded_dict[col] = encoder

with open("encoded_dict.pk", "wb") as f:
    pickle.dump(encoded_dict, f)

Bivariant analysis

In [ ]:
plt.figure(figsize=(20,15))
sns.heatmap(dataset.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation heatmap")
plt.show()

Data preprocessing

In [ ]:
# handling the outliers

def fill_outlier_values_wth_median(data, col):
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)

    IQR = q3 - q1

    lower_bound = q1 - 1.5 * IQR
    upper_bound = q3 + 1.5 * IQR

    median = data[col].median()
    data[col] = data[col].apply(lambda x : median if x < lower_bound or x > upper_bound else x)

    return data

dataset = fill_outlier_values_wth_median(dataset, "age")
dataset = fill_outlier_values_wth_median(dataset, "result")

# create subplot layout (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot for age
sns.boxplot(x=dataset['age'], ax=axes[0])
axes[0].set_title("Box Plot for Age")
axes[0].set_xlabel("Age")

# Boxplot for result
sns.boxplot(x=dataset['result'], ax=axes[1])
axes[1].set_title("Box Plot for Result")
axes[1].set_xlabel("Result")

# adjust spacing
plt.tight_layout()

# show plots
plt.show()


In [ ]:
# features and target

x = dataset.drop(columns=['Class/ASD'], axis=1)
y = dataset['Class/ASD']

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=.2, random_state=123)

print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)

print(y_train.value_counts())

In [ ]:
# smote 
smote = SMOTE(random_state=123)
x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

print(y_train_smote.value_counts()) # oversampling

In [ ]:
# model training
models = {
    "DecisionTree" : DecisionTreeClassifier(random_state=123),
    "RandomForest" : RandomForestClassifier(random_state=123),
    "XGB" : XGBClassifier(random_state=123)
}
print("Models are imported!!")

In [ ]:
# create dictionary of score
cv_scores = { }

for model_name,model in models.items():
    print(f"training {model_name} with default parameters....")
    scores = cross_val_score(model, x_train_smote, y_train_smote, cv=5, scoring="accuracy")
    cv_scores[model_name] = scores
    print(f"{model_name} cross validation accuracy {np.mean(scores):.2f}")
    print("-" * 50)
    


In [ ]:
Decision_tree = DecisionTreeClassifier(random_state=123)
Random_forest = RandomForestClassifier(random_state=123)
XGB_classifier = XGBClassifier(random_state=123)

# hyperparameter tuning

param_grid_dt = {
    "criterion": ["entropy", "gini"],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 3, 4],
    "min_samples_split": [2, 4, 6]
}

param_grid_rf = {
    "n_estimators": [50, 100, 200, 400],
    "max_depth": [4, 7, 10, 20],
    "min_samples_leaf": [1, 2, 3, 4],
    "min_samples_split": [2, 4, 6],
    "bootstrap": [True, False]
}

param_grid_xgb = {
    "n_estimators": [50, 100, 200, 400],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.1, 0.01, 0.2, 0.3],
    "subsample": [0.2, 0.55, 0.75, 1.0],
    "colsample_bytree": [0.5, 0.6, 0.7]
}

print("Models are set!!!")

In [ ]:
# hyperparameter tuning for each model
# performing the RandomizedSearchCV

# Decision Tree
random_search_dt = RandomizedSearchCV(
    estimator=Decision_tree,
    param_distributions=param_grid_dt,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=32,
    n_jobs=-1
)

# Random Forest
random_search_rf = RandomizedSearchCV(
    estimator=Random_forest,
    param_distributions=param_grid_rf,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=32,
    n_jobs=-1
)

# XGBoost
random_search_xgb = RandomizedSearchCV(
    estimator=XGB_classifier,
    param_distributions=param_grid_xgb,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=32,
    n_jobs=-1
)

# fitting models
random_search_dt.fit(x_train_smote, y_train_smote)

random_search_rf.fit(x_train_smote, y_train_smote)

random_search_xgb.fit(x_train_smote, y_train_smote)


In [ ]:
best_model = None
best_score = 0

if random_search_dt.best_score_ > best_score:
    best_model = random_search_dt.best_estimator_
    best_score = random_search_dt.best_score_
if random_search_rf.best_score_ > best_score:
    best_model = random_search_rf.best_estimator_
    best_score = random_search_rf.best_score_
if random_search_dt.best_score_ > best_score:
    best_model = random_search_xgb.best_estimator_
    best_score = random_search_xgb.best_score_

print(f"the best model is{best_model}")
print(f"The best cross validation score is {best_score}")